# Model Hub API Comparison

Orchestrate Model Hub inference APIs to analyze structural behavioral differences between a closed-source foundation model and an open-source model.

In [16]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv

## Setup

In [17]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HF_API_TOKEN = os.getenv("HF_API_TOKEN")

prompt = "Explain the difference between mutable and immutable data structures in one short sentence."

## Closed Source API

In [18]:
def query_closed_source(prompt, api_key):
    client = OpenAI(api_key=api_key)
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=60,
            temperature=0.7,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"

## Open Source API

In [19]:
def query_open_source(prompt, api_token):
    api_url = (
        "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2"
    )
    headers = {"Authorization": f"Bearer {api_token}"}
    payload = {
        "inputs": f"[INST] {prompt} [/INST]",
        "parameters": {
            "max_new_tokens": 60,
            "temperature": 0.7,
            "return_full_text": False,
        },
    }

    try:
        response = requests.post(api_url, headers=headers, json=payload)
        response.raise_for_status()
        return response.json()[0]["generated_text"].strip()
    except Exception as e:
        return f"Error: {str(e)}"

## Execution and Comparison

In [20]:
closed_output = query_closed_source(prompt, OPENAI_API_KEY)
open_output = query_open_source(prompt, HF_API_TOKEN)

print("Prompt:", prompt)
print("\n" + "=" * 50)
print("Closed-Source Model Output (OpenAI GPT-4o)")
print("=" * 50)
print(closed_output)

print("\n" + "=" * 50)
print("Open-Source Model Output (Hugging Face Mistral-7B)")
print("=" * 50)
print(open_output)

Prompt: Explain the difference between mutable and immutable data structures in one short sentence.

Closed-Source Model Output (OpenAI GPT-4o)
Error: Error code: 429 - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.

Open-Source Model Output (Hugging Face Mistral-7B)
Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("HTTPSConnection(host='api-inference.huggingface.co', port=443): Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)"))
